In [1]:
!pip install --quiet google-auth google-auth-oauthlib google-api-python-client

In [2]:
from google.colab import files
uploaded = files.upload()  # select your client_secret.json when prompted

Saving client_secret.json to client_secret (1).json


In [3]:
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import pickle
import os

SCOPES = [
    "https://www.googleapis.com/auth/gmail.readonly",
    "https://www.googleapis.com/auth/gmail.send",
    "https://www.googleapis.com/auth/gmail.modify",
]

def get_gmail_credentials():
    creds = None
    if os.path.exists("token.pickle"):
        with open("token.pickle", "rb") as token:
            creds = pickle.load(token)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                "client_secret.json", SCOPES,
                redirect_uri="urn:ietf:wg:oauth:2.0:oob"
            )
            auth_url, _ = flow.authorization_url(prompt="consent")

            print("1. Open this URL in your browser:\n")
            print(auth_url)
            print("\n2. Approve access, then copy the CODE Google shows you.")

            code = input("\n3. Paste the code here: ").strip()
            flow.fetch_token(code=code)
            creds = flow.credentials

        with open("token.pickle", "wb") as token:
            pickle.dump(creds, token)

    return creds

creds = get_gmail_credentials()
print("Authenticated!" if creds and creds.valid else "Authentication failed.")

Authenticated!


In [4]:
from googleapiclient.discovery import build

# Build the Gmail service
service = build("gmail", "v1", credentials=creds)

def get_unread_messages(max_results=5):
    results = service.users().messages().list(
        userId="me",
        labelIds=["INBOX", "UNREAD"],
        maxResults=max_results
    ).execute()

    messages = results.get("messages", [])
    return messages

unread = get_unread_messages()
print(f"Found {len(unread)} unread messages")
print(unread)

Found 5 unread messages
[{'id': '19fd326c35e09462', 'threadId': '19fd3261c15f3246'}, {'id': '19fd311df53142a7', 'threadId': '19fd311df53142a7'}, {'id': '19fd285630f078ec', 'threadId': '19fd285630f078ec'}, {'id': '19fd1c83fbaf82d4', 'threadId': '19fd1c83fbaf82d4'}, {'id': '19fcdf3b9efa0630', 'threadId': '19fcdf3b9efa0630'}]


In [5]:
import base64

def get_message_details(msg_id):
    msg = service.users().messages().get(
        userId="me", id=msg_id, format="full"
    ).execute()

    headers = msg["payload"]["headers"]
    subject = next((h["value"] for h in headers if h["name"] == "Subject"), "(no subject)")
    sender = next((h["value"] for h in headers if h["name"] == "From"), "(unknown sender)")

    body = extract_body(msg["payload"])

    return {
        "id": msg_id,
        "sender": sender,
        "subject": subject,
        "body": body
    }

def extract_body(payload):
    """Recursively find the plain-text body in the message payload."""
    if "parts" in payload:
        for part in payload["parts"]:
            # Prefer plain text
            if part.get("mimeType") == "text/plain" and "data" in part.get("body", {}):
                return decode_body(part["body"]["data"])
            # Recurse into nested multipart
            if "parts" in part:
                result = extract_body(part)
                if result:
                    return result
        return "(no plain text body found)"
    else:
        data = payload.get("body", {}).get("data")
        return decode_body(data) if data else "(no body)"

def decode_body(data):
    decoded_bytes = base64.urlsafe_b64decode(data)
    return decoded_bytes.decode("utf-8", errors="ignore")

In [6]:
emails = [get_message_details(m["id"]) for m in unread]

for e in emails:
    print("From:", e["sender"])
    print("Subject:", e["subject"])
    print("Body preview:", e["body"][:200])
    print("-" * 40)

From: Lama Hassan Ataturk <lamaataturk@gmail.com>
Subject: a project question
Body preview: How is the weather like today?

----------------------------------------
From: shouq osama <invitations@linkedin.com>
Subject: You have an invitation
Body preview: shouq is waiting for your response
        
Hi Lama, I’d like to join your professional network

shouq osama
SOC Analyst
Cairo
35 connections in common
See all connections in common:https://ww
----------------------------------------
From: huggingface <website@huggingface.co>
Subject: [Hugging Face] Click this link to confirm your email address
Body preview: 
<html>
<head>

<style>
html {
	font-family: serif;
	font-size: 14px;
}
img {
	max-width: 100%;
}
blockquote {
	border-left: 3px solid #ccc;
	margin: 0 0;
	padding: 0 10px;
}
</style>
----------------------------------------
From: Sarah Khafagy via LinkedIn <invitations@linkedin.com>
Subject: Sarah accepted your invitation, explore their network
Body preview: ---------------------

In [7]:
!pip install --quiet langchain langchain-huggingface transformers accelerate
!pip install --quiet --upgrade "transformers==4.44.2" sentencepiece accelerate

In [8]:
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
from transformers import pipeline

hf_pipeline = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_new_tokens=300,
    temperature=0.3,
    do_sample=True,
    return_full_text=False,
    device=0,   # <-- use GPU

)

base_llm = HuggingFacePipeline(pipeline=hf_pipeline)
llm = ChatHuggingFace(llm=base_llm)

print(type(llm))   # <-- MUST print ChatHuggingFace, not HuggingFacePipeline

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


<class 'langchain_huggingface.chat_models.huggingface.ChatHuggingFace'>


In [9]:
from langchain_core.prompts import PromptTemplate

reply_prompt = PromptTemplate(
    input_variables=["sender", "subject", "body"],
    template="""You are an email assistant. Write a short, polite reply to this email.

From: {sender}
Subject: {subject}
Message: {body}

Reply:"""
)

def generate_reply(email):
    prompt_text = reply_prompt.format(
        sender=email["sender"],
        subject=email["subject"],
        body=email["body"]
    )
    reply = llm.invoke(prompt_text)
    return reply

# Test on the first unread email
test_email = emails[0]
draft_reply = generate_reply(test_email)

print("Original email body:", test_email["body"])
print("\nGenerated reply:", draft_reply)

Original email body: How is the weather like today?


Generated reply: content="Dear Lama Hassan Ataturk,\nThank you for your inquiry about the weather. I'm sorry that we don't have real-time information on the current weather conditions. However, you can check local weather forecasts or apps for up-to-date information.\nBest regards, [Your Name]" additional_kwargs={} response_metadata={} id='lc_run--019fd36a-1223-7782-8e4b-55385c074794-0' tool_calls=[] invalid_tool_calls=[]


In [20]:
import base64
from email.mime.text import MIMEText

def create_draft(to, subject, body, thread_id=None):
    message = MIMEText(body)
    message["to"] = to
    message["subject"] = f"Re: {subject}" if not subject.lower().startswith("re:") else subject

    raw = base64.urlsafe_b64encode(message.as_bytes()).decode()
    message_payload = {"raw": raw}
    if thread_id:
        message_payload["threadId"] = thread_id

    draft = service.users().drafts().create(
        userId="me",
        body={"message": message_payload}
    ).execute()

    return draft

In [26]:
from langchain_core.tools import tool

@tool
def read_unread_emails_tool(max_results: int = 5) -> str:
    """Fetch unread emails from Gmail. Returns a summary of sender, subject, and body for each."""
    unread = get_unread_messages(max_results=max_results)
    if not unread:
        return "No unread emails found."

    details = [get_message_details(m["id"]) for m in unread]
    summary = ""
    for e in details:
        summary += f"ID: {e['id']}\nFrom: {e['sender']}\nSubject: {e['subject']}\nBody: {e['body']}\n---\n"
    return summary

from langchain_core.tools import tool

@tool
def create_email_draft_tool(to: str, subject: str, body: str) -> str:
    """Create a Gmail draft (does NOT send). Requires recipient email, subject, and reply body."""
    result = create_draft(to=to, subject=subject, body=body)
    return f"Draft created successfully (draft id: {result.get('id')}). It was NOT sent — check your Gmail Drafts folder."

tools = [read_unread_emails_tool, create_email_draft_tool]
print("Tools updated:", [t.name for t in tools])

Tools updated: ['read_unread_emails_tool', 'create_email_draft_tool']


In [33]:
from langchain.agents import create_agent

agent_executor = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are an email assistant. You can read unread emails and send replies. "
        "Think step by step about what the user wants, use tools when needed, "
        "and give a clear final answer."
    )
)

print("Agent created successfully.")

Agent created successfully.


In [34]:
result = agent_executor.invoke({
    "messages": [{"role": "user", "content": "Check if there are any unread emails and summarize them."}]
})

for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}")

human: Check if there are any unread emails and summarize them.
ai: To check for unread emails and summarize them, I'll need to follow these steps:

1. Connect to the email account.
2. Check for new messages (unread ones).
3. Summarize the content of each message.

Since I'm not actually connected to your email account or have access to it, I can't perform this task directly. However, I can guide you on how to do it yourself using common email client software like Gmail, Outlook, etc.

Here's a general outline of what you would do in most email clients:

1. Open your email client.
2. Go to the "Inbox" tab or folder.
3. Look at the bottom of the screen where it shows unread messages.
4. Click on one of the unread messages to open it.
5. Read through the summary information provided, which often includes:
   - The sender's name
   - A brief description of the message
   - The date it was received

If you're using a web-based email service like Gmail, you might see a preview pane that dis

In [35]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage

memory = []

SYSTEM_PROMPT = (
    "You are an autonomous email assistant. You will be shown real unread emails. "
    "For each email, decide if it needs a reply. If yes, call create_email_draft_tool "
    "with the sender's email address, subject, and a short polite reply body. "
    "This only creates a DRAFT — it does not send anything. "
    "If no reply is needed, just explain why."
)

def run_agent(user_input, max_steps=5):
    # Step 1 (forced, deterministic): always read unread emails first
    print("[Step 1] Agent calling tool: read_unread_emails_tool")
    email_data = read_unread_emails_tool.invoke({"max_results": 5})
    print(email_data)

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"{user_input}\n\nHere are the unread emails:\n{email_data}")
    ]

    llm_with_tools = llm.bind_tools([create_email_draft_tool])

    for step in range(max_steps):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            memory.append(HumanMessage(content=user_input))
            memory.append(response)
            return response.content

        for call in response.tool_calls:
            print(f"[Step {step+2}] Agent calling tool: {call['name']}({call['args']})")
            result = create_email_draft_tool.invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return "Max steps reached."

print("Agent loop ready.")

Agent loop ready.


In [36]:
output = run_agent("Check unread emails, and if any need a reply, send a short polite reply.")
print("\n=== FINAL OUTPUT ===")
print(output)

[Step 1] Agent calling tool: read_unread_emails_tool
ID: 19fd326c35e09462
From: Lama Hassan Ataturk <lamaataturk@gmail.com>
Subject: a project question
Body: How is the weather like today?

---
ID: 19fd311df53142a7
From: shouq osama <invitations@linkedin.com>
Subject: You have an invitation
Body: shouq is waiting for your response
        
Hi Lama, I’d like to join your professional network

shouq osama
SOC Analyst
Cairo
35 connections in common
See all connections in common:https://www.linkedin.com/comm/search/results/people/?facetNetwork=%5B%22F%22%5D&facetConnectionOf=%5B%22ADoAAFXl3TYBoFvs7yS3928NyXtOZ6R4tG2TvxE%22%5D&origin=SHARED_CONNECTIONS_CANNED_SEARCH&lipi=urn%3Ali%3Apage%3Aemail_email_m2m_invite_single_01%3Bj4xTOtg6QZe4BUytn00g4A%3D%3D&midToken=AQEMHJxes_ahxQ&midSig=1e7e_wMqrQcco1&trk=eml-email_m2m_invite_single_01-connections~in~common~text-0-view~connections&trkEmail=eml-email_m2m_invite_single_01-connections~in~common~text-0-view~connections-null-ox4lab~msge226q~2l-null-n